# 🏥 Civil Sahai — LoRA / QLoRA Fine-Tuning Google Gemma 2 (2B-IT)
### Multilingual Clinical Intake Synthesizer & Emergency Handover System for Public Healthcare

This notebook demonstrates how to fine-tune **Google Gemma 2 (2B-IT)** using **4-bit QLoRA** and **TRL SFTTrainer** to:
1. Convert unstructured patient descriptions in **Gujarati, Hindi, and English** into standardized clinical intake JSON.
2. Synthesize rural PHC **Inter-Hospital Emergency Referral Chits** into SBAR handover packets and identify **Critical Handover Gaps**.
3. Enforce **Zero-Diagnosis Responsible AI safety guardrails**.

## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets torch

## ⚡ Step 2: Verify GPU Environment

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Running on CPU. Please enable GPU Accelerator (T4 or P100) in Kaggle/Colab settings.")

## 🏥 Step 3: Define Clinical Training Dataset & Prompts

In [ ]:
import json
from datasets import Dataset

GEMMA_INTAKE_SYSTEM_PROMPT = """You are a specialized multilingual clinical intake assistant powered by Google Gemma for Civil Hospital OPD.
Convert patient descriptions in Gujarati, Hindi, English, or mixed languages into structured clinical intake data.

SAFETY RULES:
1. NEVER provide medical diagnosis.
2. NEVER suggest medicines or dosages.
3. Extract only facts directly stated by the patient.

Output STRICTLY JSON with this schema:
{
  "language_detected": "Gujarati | Hindi | English | Mixed",
  "chief_complaint": "Brief primary reason for visit in English",
  "symptoms": ["list of reported symptoms translated to clinical English"],
  "duration": "Duration of symptoms",
  "age": "Patient age or Not specified",
  "gender": "Patient gender or Not specified",
  "existing_conditions": ["Chronic conditions or None reported"],
  "current_medicines": ["Current medications or None reported"],
  "allergies": "Reported allergies or Not specified",
  "missing_details": ["List missing fields that staff should ask"],
  "emergency_indicators": ["Emergency red flags or None detected"],
  "doctor_summary": "Concise objective 2-3 sentence summary for the doctor without any diagnosis."
}"""

GEMMA_TRANSFER_SYSTEM_PROMPT = """You are an emergency inter-hospital handover specialist powered by Google Gemma for Civil Hospital Emergency Trauma Center.
Convert patient transfer notes or referral chits from rural PHCs/CHCs into a structured Emergency Transfer Handover Packet.

SAFETY RULES:
1. NO medical diagnosis or prescription.
2. Capture referral details, pre-transfer interventions, and transit events accurately.
3. Identify CRITICAL HANDOVER GAPS (vital facts city emergency doctors frequently miss).

Output STRICTLY JSON with this schema:
{
  "language_detected": "Gujarati | Hindi | English | Mixed",
  "patient_name": "Patient name or Unknown",
  "age_gender": "Age and Gender",
  "referring_facility": "Name of village PHC/CHC",
  "receiving_facility": "Civil Hospital Emergency Trauma Center",
  "transfer_reason": "Clinical justification for referral",
  "chief_condition_at_referral": "Primary acute condition",
  "symptoms": ["Key symptoms"],
  "pre_transfer_treatments": ["List of medications/injections/fluids given before transfer"],
  "transit_events": ["Events during transit or Stable in transit"],
  "allergies": "Known allergies or Unverified",
  "critical_handover_gaps": ["Crucial missing transfer details"],
  "emergency_red_flags": ["Immediate life-threat alerts"],
  "doctor_handover_summary": "3-4 sentence SBAR summary for receiving doctor."
}"""

# Sample Regional Dataset
clinical_dataset = [
    {
        "instruction_type": "clinical_intake",
        "input": "મને બે દિવસથી છાતીમાં દુખે છે અને આજે સવારથી શ્વાસ લેવામાં તકલીફ થાય છે. બ્લડ પ્રેશરની દવા ચાલે છે પણ બે દિવસથી લીધી નથી. ઉંમર 54 વર્ષ છે.",
        "output": {
            "language_detected": "Gujarati",
            "chief_complaint": "Acute chest pain and severe dyspnea for 2 days",
            "symptoms": ["Chest pain", "Shortness of breath / Dyspnea"],
            "duration": "2 days (exacerbated this morning)",
            "age": "54 years",
            "gender": "Not specified",
            "existing_conditions": ["Hypertension (High BP)"],
            "current_medicines": ["Antihypertensive medication (missed for past 2 days)"],
            "allergies": "Not specified",
            "missing_details": ["Current blood pressure reading", "Radiation of chest pain to left arm or jaw"],
            "emergency_indicators": ["Potential Acute Coronary Syndrome (ACS)", "Acute Dyspnea"],
            "doctor_summary": "54-year-old patient with known hypertension presenting with a 2-day history of chest pain and worsening dyspnea after missing BP medication. High suspicion of acute cardiovascular or respiratory compromise requiring urgent triage."
        }
    },
    {
        "instruction_type": "clinical_intake",
        "input": "नमस्ते डॉक्टर साहब, मुझे 4 दिन से लगातार खांसी है और बलगम में थोड़ा खून आया है। शाम को पसीना आता है। उम्र 39 साल। कोई पुरानी बीमारी नहीं है।",
        "output": {
            "language_detected": "Hindi",
            "chief_complaint": "Persistent cough with hemoptysis and night sweats for 4 days",
            "symptoms": ["Persistent productive cough", "Hemoptysis (blood in sputum)", "Nocturnal diaphoresis (night sweats)"],
            "duration": "4 days",
            "age": "39 years",
            "gender": "Not specified",
            "existing_conditions": ["None reported"],
            "current_medicines": ["None reported"],
            "allergies": "Not specified",
            "missing_details": ["Chest X-Ray history", "History of tuberculosis exposure"],
            "emergency_indicators": ["Hemoptysis (Red Flag for Pulmonary pathology)"],
            "doctor_summary": "39-year-old patient with no prior chronic illness presenting with a 4-day history of cough, blood-tinged sputum, and night sweats. Pulmonary evaluation and infectious workup indicated."
        }
    },
    {
        "instruction_type": "rural_transfer",
        "input": "PHC Bavla Referral Chit: 42 F transferred to Civil Hospital Ahmedabad. Farm worker, Viper snake bite on right ankle at 8:30 AM. Local swelling ++, bleeding from bite puncture. 2 vials polyvalent ASV given IV with Normal Saline at 9:15 AM. Tetanus Toxoid 0.5ml given. In 108 ambulance, patient had 1 episode gum bleeding. BP 100/60, Pulse 108.",
        "output": {
            "language_detected": "Gujarati / Mixed",
            "patient_name": "Unknown (Female, 42y)",
            "age_gender": "42 Female",
            "referring_facility": "PHC Bavla",
            "receiving_facility": "Civil Hospital Emergency Trauma Center",
            "transfer_reason": "Hemotoxic Viper snake envenomation with active coagulopathy",
            "chief_condition_at_referral": "Snake bite on right ankle with systemic envenomation",
            "symptoms": ["Local right ankle edema", "Puncture bleeding", "Systemic gum bleeding (gingival hemorrhage)", "Tachycardia (Pulse 108)"],
            "pre_transfer_treatments": ["2 vials Polyvalent ASV IV (9:15 AM)", "IV Normal Saline", "Tetanus Toxoid 0.5ml IM"],
            "transit_events": ["1 episode of spontaneous gum bleeding in 108 ambulance"],
            "allergies": "Unverified",
            "critical_handover_gaps": ["20-minute Whole Blood Clotting Test (20WBCT) result missing", "Urine output in transit not documented"],
            "emergency_red_flags": ["Systemic hemotoxicity / DIC risk (Gum bleeding)", "Requirement for immediate additional ASV titration"],
            "doctor_handover_summary": "42-year-old female transferred from PHC Bavla following a Viper bite with 2 vials of ASV pre-administered. Developed in-transit gum bleeding indicating evolving coagulopathy. Urgent repeat 20WBCT, coagulation profile, and escalated ASV protocol indicated immediately upon arrival."
        }
    }
]

def format_entry(item):
    is_transfer = item.get("instruction_type") == "rural_transfer"
    sys_prompt = GEMMA_TRANSFER_SYSTEM_PROMPT if is_transfer else GEMMA_INTAKE_SYSTEM_PROMPT
    input_label = "Referral Chit:" if is_transfer else "Patient Input:"
    target_json_str = json.dumps(item["output"], indent=2, ensure_ascii=False)
    
    formatted_text = (
        f"<start_of_turn>user\n"
        f"{sys_prompt}\n\n"
        f"{input_label}\n\"\"\"\n{item['input']}\n\"\"\"\n\n"
        f"JSON Output:<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{target_json_str}<end_of_turn>"
    )
    return {"text": formatted_text}

raw_ds = Dataset.from_list(clinical_dataset)
formatted_ds = raw_ds.map(format_entry)
print(f"✓ Prepared {len(formatted_ds)} formatted training samples.")
print("\n--- Sample Prompt Template ---")
print(formatted_ds[0]["text"][:450] + "...")

## 🤖 Step 4: Load Google Gemma 2 (2B-IT) in 4-bit (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-2-2b-it"

print(f"Loading Tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-Bit NF4 Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} in 4-bit on GPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)
    base_model.config.use_cache = False

print("✓ Base model loaded successfully!")

## 🎯 Step 5: Configure LoRA Adapter

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 🚀 Step 6: Train with TRL SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir="./civil_sahai_gemma2_lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    logging_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=not use_bf16 and torch.cuda.is_available(),
    bf16=use_bf16,
    max_grad_norm=0.3,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_ds,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting LoRA fine-tuning on Google Gemma 2B...")
trainer.train()

# Save trained adapters
ADAPTER_OUTPUT_DIR = "./civil_sahai_gemma2_lora/final_adapter"
trainer.model.save_pretrained(ADAPTER_OUTPUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)
print(f"✓ LoRA Adapters saved to: {ADAPTER_OUTPUT_DIR}")

## 🩺 Step 7: Test Fine-Tuned Model Inference

In [ ]:
def run_inference(patient_text, is_transfer=False):
    sys_prompt = GEMMA_TRANSFER_SYSTEM_PROMPT if is_transfer else GEMMA_INTAKE_SYSTEM_PROMPT
    input_label = "Referral Chit:" if is_transfer else "Patient Input:"
    
    prompt = (
        f"<start_of_turn>user\n"
        f"{sys_prompt}\n\n"
        f"{input_label}\n\"\"\"\n{patient_text}\n\"\"\"\n\n"
        f"JSON Output:<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=450, temperature=0.1)
        
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

# Test 1: Gujarati Patient Walk-in
test_input = "મને 3 દિવસથી ખૂબ તાવ છે, માથું દુખે છે અને ચક્કર આવે છે. ઉંમર 45 છે."
print("=== TEST CASE 1: GUJARATI WALK-IN INTAKE ===")
print(run_inference(test_input, is_transfer=False))

# Test 2: Rural Inter-Hospital Referral Chit
test_transfer = "PHC Sanand Referral: 58 M, crushing chest pain since 6 AM. Sublingual Sorbitrate given at 6:45 AM. BP 170/100. Transferred via 108 ambulance."
print("\n=== TEST CASE 2: RURAL TRANSFER DOSSIER ===")
print(run_inference(test_transfer, is_transfer=True))

## 📤 Step 8: Deploy Fine-Tuned Model to Ollama
To use your fine-tuned LoRA adapters with Ollama in the Civil Sahai Web App:
1. Merge LoRA weights into the base Gemma model:
   ```python
   from peft import PeftModel
   base = AutoModelForCausalLM.from_pretrained('google/gemma-2-2b-it', torch_dtype=torch.float16)
   merged = PeftModel.from_pretrained(base, './civil_sahai_gemma2_lora/final_adapter').merge_and_unload()
   merged.save_pretrained('./merged_gemma2_civil_sahai')
   ```
2. Convert to GGUF using `llama.cpp`:
   ```bash
   python convert-hf-to-gguf.py ./merged_gemma2_civil_sahai --outfile gemma2-civil-sahai.gguf
   ```
3. Create an Ollama `Modelfile`:
   ```dockerfile
   FROM ./gemma2-civil-sahai.gguf
   PARAMETER temperature 0.1
   PARAMETER top_p 0.9
   ```
4. Register with Ollama:
   ```bash
   ollama create civil-sahai -f Modelfile
   ollama run civil-sahai
   ```